# 01 — Exploratory data analysis (XJTU-SY Bearing1_*)

**Hybrid quantum–classical RUL (rolling bearings)** — notebook to inspect raw vibration, degradation-sensitive quantities, spectra, feature redundancy, and cross-run degradation shape.

This notebook assumes the dataset lives under `data/raw/` and uses `src/preprocessing.py` and `src/features.py`. Full loads are large (millions of samples per bearing); interactive plots use **downsampled** time series where needed so Plotly remains responsive.


## Environment and imports

We add `src/` to the path, import preprocessing/feature utilities, and configure Plotly for inline display in Jupyter.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Project root: repo root whether kernel cwd is repo or notebooks/
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").is_dir() else NOTEBOOK_DIR.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from preprocessing import split_bearings
from features import extract_all_features

DATA_RAW = PROJECT_ROOT / "data" / "raw"

pd.options.display.max_columns = 20
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


## 1. Load all five bearings

`split_bearings` resolves `Bearing1_1` … `Bearing1_5` under `data/raw` (or a parent path) and returns training bearings 1–4 plus test bearing 5. We concatenate them here in run order **1 → 5** for consistent plotting.

**Expectation:** records are long run-to-failure trajectories; loading can take several minutes depending on disk speed.


In [ ]:
bearing_names = [f"Bearing1_{i}" for i in range(1, 6)]

train_list, test_list = split_bearings(DATA_RAW)
bearings_ordered = train_list + test_list
assert len(bearings_ordered) == 5

for name, df in zip(bearing_names, bearings_ordered):
    print(f"{name}: n={len(df):,} samples, columns={list(df.columns)}")


## 2. Raw vibration (horizontal + vertical)

Full-resolution plotting is impractical in the browser (tens of millions of points). We **decimate** each channel with a stride chosen so each channel uses at most ~80k points and plot against sample index so early-run behavior is easy to inspect.

**Finding:** healthy segments usually show bounded oscillation; degradation often coincides with growing impulses and elevated amplitude—use Plotly zoom/pan to compare horizons within and across bearings.


In [ ]:
MAX_POINTS = 80_000


def decimate_slice(n: int, max_points: int = MAX_POINTS) -> slice:
    if n <= max_points:
        return slice(None)
    step = int(np.ceil(n / max_points))
    return slice(0, n, step)


fig = make_subplots(
    rows=5,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.05,
    subplot_titles=[f"{n} — raw (decimated)" for n in bearing_names],
)

colors_h = px.colors.qualitative.Set2
colors_v = px.colors.qualitative.Pastel2

for row, (name, df) in enumerate(zip(bearing_names, bearings_ordered), start=1):
    sl = decimate_slice(len(df))
    t = np.arange(len(df))[sl]
    h = df["horizontal_acc"].to_numpy()[sl]
    v = df["vertical_acc"].to_numpy()[sl]

    fig.add_trace(
        go.Scatter(
            x=t,
            y=h,
            mode="lines",
            name=f"{name} H",
            legendgroup=name + "H",
            line=dict(width=0.8, color=colors_h[(row - 1) % len(colors_h)]),
            showlegend=(row == 1),
        ),
        row=row,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=t,
            y=v,
            mode="lines",
            name=f"{name} V",
            legendgroup=name + "V",
            line=dict(width=0.8, color=colors_v[(row - 1) % len(colors_v)]),
            showlegend=(row == 1),
        ),
        row=row,
        col=1,
    )

fig.update_layout(
    height=1400,
    title_text="Raw acceleration (decimated sample index)",
    hovermode="x unified",
)
fig.update_xaxes(title_text="sample index (decimated)")
fig.update_yaxes(title_text="acc (a.u.)")
fig.show()


## 3. RMS over time (degradation trend)

We compute the **rolling RMS** on the horizontal channel with window **1024** (aligned with the feature extractor) and subsample for plotting. Rising RMS in later life is a simple proxy for accumulating fault energy.

**Finding:** onset and steepness differ across runs; the held-out test bearing should be interpreted in the same units when reporting generalization.


In [ ]:
WIN = 1024


def rolling_rms(x: pd.Series, window: int = WIN) -> pd.Series:
    return x.pow(2).rolling(window, min_periods=window).mean().pow(0.5)


def subsample_series(y: np.ndarray, max_points: int = 25_000):
    n = len(y)
    if n <= max_points:
        return np.arange(n), y
    step = int(np.ceil(n / max_points))
    idx = np.arange(0, n, step)
    return idx, y[idx]


fig = go.Figure()
palette = px.colors.qualitative.Bold

for i, (name, df) in enumerate(zip(bearing_names, bearings_ordered)):
    rms_h = rolling_rms(df["horizontal_acc"]).to_numpy()
    mask = np.isfinite(rms_h)
    t = np.arange(len(rms_h))[mask]
    y = rms_h[mask]
    ix, ys = subsample_series(y)
    fig.add_trace(
        go.Scatter(
            x=t[ix],
            y=ys,
            mode="lines",
            name=f"{name} rolling RMS (H)",
            line=dict(width=1.2, color=palette[i % len(palette)]),
        )
    )

fig.update_layout(
    title="Rolling RMS (horizontal), window=1024",
    xaxis_title="sample index",
    yaxis_title="RMS",
    hovermode="x unified",
    height=520,
)
fig.show()


## 4. FFT spectrum at three lifecycle stages

For each bearing we extract **1024-sample** windows at:

- **Early:** beginning of the record  
- **Middle:** centered (start index `(N − 1024) // 2`)  
- **Near-failure:** last full window  

Each window is mean-removed before `rfft`. The horizontal axis is **cycles per sample** (digital frequency; Nyquist = 0.5). Multiply by the sampling rate in Hz if you need physical frequency.

**Finding:** near-failure spectra often show elevated high-frequency energy or peak reshaping relative to early life on the same unit.


In [ ]:
SEG = 1024


def stage_starts(n: int, seg: int = SEG):
    if n < seg:
        raise ValueError("record shorter than FFT window")
    early = 0
    mid = (n - seg) // 2
    late = n - seg
    return early, mid, late


fig = make_subplots(
    rows=5,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=[f"{n} — |FFT| (mean-removed horizontal)" for n in bearing_names],
)

st_colors = {"early": "#3366cc", "middle": "#109618", "near_failure": "#dc3912"}

for row, (name, df) in enumerate(zip(bearing_names, bearings_ordered), start=1):
    n = len(df)
    e, m, late = stage_starts(n)
    h_sig = df["horizontal_acc"].to_numpy(dtype=float)

    for label, start in [("early", e), ("middle", m), ("near_failure", late)]:
        w = h_sig[start : start + SEG]
        w = w - np.mean(w)
        spec = np.abs(np.fft.rfft(w))
        freqs = np.fft.rfftfreq(SEG, d=1.0)
        fig.add_trace(
            go.Scatter(
                x=freqs,
                y=spec,
                mode="lines",
                name=label,
                legendgroup=label,
                line=dict(width=1.0, color=st_colors[label]),
                showlegend=(row == 1),
            ),
            row=row,
            col=1,
        )

fig.update_layout(
    height=1600,
    title_text="FFT magnitude — three windows per bearing",
    hovermode="x unified",
)
fig.update_xaxes(title_text="frequency (cycles/sample)")
fig.update_yaxes(title_text="|RFFT|")
fig.show()


## 5. Correlation heatmap of extracted features

We slide **1024 / 512** windows (`extract_all_features`), **pool** rows from all five bearings, and compute the Pearson correlation of numeric feature columns (meta columns excluded). Strong positive blocks (e.g., scale-related time statistics) indicate redundancy; near-diagonal structure is expected after stacking `h_` and `v_` channels.

**Finding:** correlation structure informs regularization, feature selection, and how aggressively inputs can be compressed before a hybrid quantum–classical head.


In [ ]:
WINDOW = 1024
STRIDE = 512

feat_parts = []
for name, df in zip(bearing_names, bearings_ordered):
    fdf = extract_all_features(df, window_size=WINDOW, stride=STRIDE)
    fdf = fdf.assign(bearing=name)
    feat_parts.append(fdf)

Feat = pd.concat(feat_parts, axis=0, ignore_index=True)
feature_cols = [c for c in Feat.columns if c not in ("window_start", "window_end", "bearing")]

corr = Feat[feature_cols].corr(numeric_only=True)

short_labels = [
    (c[:22] + "…") if len(c) > 22 else c for c in corr.columns
]

fig = go.Figure(
    data=go.Heatmap(
        z=corr.values,
        x=short_labels,
        y=short_labels,
        colorscale="RdBu",
        zmid=0,
        zmin=-1,
        zmax=1,
        colorbar=dict(title="Pearson r"),
    )
)
fig.update_layout(
    title="Feature correlation — pooled windows (all bearings)",
    xaxis=dict(tickangle=-65, tickfont=dict(size=7)),
    yaxis=dict(tickfont=dict(size=7)),
    height=950,
    width=1020,
    margin=dict(l=130, r=40, t=70, b=170),
)
fig.show()


## 6. Comparative degradation curves (all bearings, one figure)

We map sample index **k** to **normalized life** τ = k / (N − 1). The plotted quantity is horizontal **rolling RMS** (window 1024) divided by the **median RMS over the first 5%** of samples (after the rolling window becomes valid). Curves therefore start near unity and late-life growth is directly comparable across runs of different lengths.

**Finding:** cross-bearing spread illustrates why fixed thresholds rarely generalize; supervised RUL targets (e.g., from `compute_rul`) encode a smooth idealization of this progression.


In [ ]:
def normalized_rms_curve(df: pd.DataFrame, win: int = WIN, early_frac: float = 0.05):
    n = len(df)
    rms = rolling_rms(df["horizontal_acc"]).to_numpy()
    idx_all = np.arange(len(rms))
    ok = np.isfinite(rms)
    idx = idx_all[ok]
    if len(idx) == 0:
        return np.array([]), np.array([])
    rms_ok = rms[ok]
    tau = idx.astype(float) / max(1, n - 1)

    warm = int(max(win, early_frac * n))
    base_mask = idx < warm
    baseline = np.median(rms_ok[base_mask]) if np.any(base_mask) else np.median(rms_ok[:500])
    if baseline < 1e-12:
        baseline = 1e-12
    return tau, rms_ok / baseline


fig = go.Figure()
for i, (name, df) in enumerate(zip(bearing_names, bearings_ordered)):
    tau, y = normalized_rms_curve(df)
    if len(tau) == 0:
        continue
    if len(tau) > 12_000:
        step = int(np.ceil(len(tau) / 12_000))
        tau, y = tau[::step], y[::step]
    fig.add_trace(
        go.Scatter(
            x=tau,
            y=y,
            mode="lines",
            name=name,
            line=dict(width=1.6, color=palette[i % len(palette)]),
        )
    )

fig.update_layout(
    title="Comparative degradation — RMS(H) / median early RMS",
    xaxis_title="normalized life τ ∈ [0, 1]",
    yaxis_title="rolling RMS / baseline",
    hovermode="x unified",
    height=520,
)
fig.show()


## Summary

| Section | Takeaway |
|---------|----------|
| Raw | Decimation preserves shape for browser-scale interactive inspection. |
| Rolling RMS | Simple, interpretable trend; window matches feature settings. |
| FFT stages | Early/middle/late spectra make spectral fault evolution visible. |
| Correlation | Pooled windows reveal redundancy for modeling choices. |
| Overlay | Normalized-time RMS compares heterogeneous run lengths fairly. |

Proceed to `02_preprocessing.ipynb` to cache windowed datasets, labels, and scalers for training.
